# Customer Churn: Exploratory Data Analysis (Phase 2)
For every chart write down: **question -> what the chart shows -> business insight**. Use the empty markdown cells under each chart for your own notes.

In [ ]:
import sys
sys.path.append("..")          # so we can import from src/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_data
from src.preprocessing import clean_data

sns.set_theme(style="whitegrid")
raw = load_data("../data/Telco-Customer-Churn.csv")
df = clean_data(raw)                       # Churn is now 0/1
eda = df.copy()
eda["ChurnLabel"] = eda["Churn"].map({0: "No", 1: "Yes"})   # readable labels for plots
eda.shape

## 1. Churn distribution
**Question:** how imbalanced is the target?

In [ ]:
sns.countplot(data=eda, x="ChurnLabel")
plt.title("Churn distribution")
plt.show()
print((eda["Churn"].value_counts(normalize=True) * 100).round(1))

**My notes:** 

## 2. Churn RATE by category
We plot the churn *rate* (churners / group size), not raw counts, so groups of different sizes compare fairly. The red line is the overall churn rate.

In [ ]:
def churn_rate_plot(col, rotate=0):
    rate = eda.groupby(col)["Churn"].mean().mul(100).sort_values(ascending=False)
    ax = sns.barplot(x=rate.index, y=rate.values)
    ax.axhline(eda["Churn"].mean() * 100, color="red", linestyle="--", label="Overall churn %")
    ax.set_ylabel("Churn rate (%)")
    ax.set_title(f"Churn rate by {col}")
    plt.xticks(rotation=rotate)
    plt.legend()
    plt.show()
    print(rate.round(1))

for col in ["gender", "SeniorCitizen", "Contract", "InternetService"]:
    churn_rate_plot(col)
churn_rate_plot("PaymentMethod", rotate=15)

**My notes (gender / senior / contract / internet / payment):** 

## 3. Numeric features vs churn
Histogram shows the shape, box plot shows median and spread.

In [ ]:
for col in ["tenure", "MonthlyCharges"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=eda, x=col, hue="ChurnLabel", bins=30, element="step", ax=axes[0])
    sns.boxplot(data=eda, x="ChurnLabel", y=col, ax=axes[1])
    axes[0].set_title(f"{col} distribution by churn")
    axes[1].set_title(f"{col} by churn (box plot)")
    plt.show()

**My notes (tenure / monthly charges):** 

## 4. Correlation heatmap (numeric columns only)
Correlation is not causation.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "NumServices", "Churn"]
sns.heatmap(eda[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

**My notes:** 

## 5. Exercises
1. Churn rate by `Partner` and `Dependents`.
2. Tenure buckets with `pd.cut`.
3. Churn rate for Month-to-month AND Fiber optic together.

In [ ]:
# Exercise 1
for col in ["Partner", "Dependents"]:
    churn_rate_plot(col)

In [ ]:
# Exercise 2: tenure buckets
eda["TenureGroup"] = pd.cut(eda["tenure"], [-1, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"])
churn_rate_plot("TenureGroup")

In [ ]:
# Exercise 3: two conditions together
mask = (eda["Contract"] == "Month-to-month") & (eda["InternetService"] == "Fiber optic")
print("Customers in this group:", mask.sum())
print("Churn rate: %.1f%%" % (eda.loc[mask, "Churn"].mean() * 100))